## EDA Summary — decisions carried to feature engineering/selection

**Target:**
- drop outliers, best to drop up to 10th percentile and above 99th percentile

**Drop:**
- model (30k+ uniques)
- size (70% null, flat across categories)
- paint_color (30% null, flat across categories)
- title_status='lien' rows (leakage) — or drop title_status entirely if imbalance too severe
- cars before year 2000 (heavy skew, low counts, weak relationship)
- odometer outliers (impossible values)

**Engineer:**
- age = current_year - year
- location_cluster from lat/long (KMeans)
- bucket rare categories into "other"

**Class imbalance:**
- manufacturer, condition, cylinders, gas, transmission, fuel, type
- state (redundancy vs lat/long/region, but better to keep for now)

**Keep as-is:**
- drive

**Open questions for next notebook:**
- state vs lat/long/region redundancy
- best encoding strategy for region (400 categories)

**Mutual info results:**
- odometer, lat, long, year, transmission, condition, drive, type - STRONG

# Minimal Data Preparation for Baseline Eval

In [24]:
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
    
warnings.filterwarnings("ignore")

sns.set_style("darkgrid")

# Load the cleaned dataset
data_original = pd.read_parquet(".cache/vehicles_cleaned.parquet")
data = data_original.copy()

from car_pipeline import OutlierDetector

In [25]:
data.head()

,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,size,type,paint_color,state,lat,long
0,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN,NaN
1,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN,NaN
2,florida keys,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl,NaN,NaN
3,worcester / central MA,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma,NaN,NaN
4,greensboro,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc,NaN,NaN


In [26]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 18 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   region        426880 non-null  str    
 1   price         426880 non-null  int64  
 2   year          425675 non-null  float64
 3   manufacturer  409234 non-null  str    
 4   model         421603 non-null  str    
 5   condition     252776 non-null  str    
 6   cylinders     249202 non-null  str    
 7   fuel          423867 non-null  str    
 8   odometer      422480 non-null  float64
 9   title_status  418638 non-null  str    
 10  transmission  424324 non-null  str    
 11  drive         296313 non-null  str    
 12  size          120519 non-null  str    
 13  type          334022 non-null  str    
 14  paint_color   296677 non-null  str    
 15  state         426880 non-null  str    
 16  lat           420331 non-null  float64
 17  long          420331 non-null  float64
dtypes: float64(4), 

## Target Variable Outlier Handling

In [27]:
# Take only the cars priced 0$ and describe them, and add .nunique for unique values in the describe method

data_zero_price = data[data['price'] == 0]
data_zero_price.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
region,32895,383,salem,780,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,32895.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
year,32863.0,NaN,NaN,NaN,2013.973344,6.3591,1903.0,2012.0,2015.0,2017.0,2022.0
manufacturer,31434,40,ford,6130,NaN,NaN,NaN,NaN,NaN,NaN,NaN
model,32319,4172,f-150,893,NaN,NaN,NaN,NaN,NaN,NaN,NaN
condition,10180,6,excellent,6542,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cylinders,15627,8,6 cylinders,5322,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fuel,32476,5,gas,25778,NaN,NaN,NaN,NaN,NaN,NaN,NaN
odometer,30785.0,NaN,NaN,NaN,89312.677408,297007.173989,0.0,34908.0,72000.0,110000.0,10000000.0
title_status,32387,6,clean,32250,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
nonzero_price = data.loc[data['price'] > 0, 'price']

# Inspect where the distribution stops looking like junk
for q in [0.001, 0.005, 0.01, 0.02, 0.025, 0.03, 0.05]:
    print(f"{q*100}th pct: {nonzero_price.quantile(q):,.0f}")


0.1th pct: 1
0.5th pct: 3
1.0th pct: 150
2.0th pct: 396
2.5th pct: 500
3.0th pct: 708
5.0th pct: 1,988


In [29]:
# Function to find suspicious car prices based on a cutoff value, where cutoff is a list of values,
# it returns the percentage of suspicious prices for each cutoff value, each percentage printed
# next to its cutoff in the same row.

def find_suspicious_price(data, cutoff):
    
    suspicious_price_pct = []
    pct_cutoff_dict = {}
    for i in cutoff:
        suspicious_price = (
            (data['price'] > 0) & (data['price'] < i) &
            (data['odometer'] < 150000) &          # not obviously a wreck
            (data['condition'].isin(['excellent', 'good', 'like new']))
        )
        suspicious_price_pct.append(suspicious_price.sum() / len(data.loc[data['price'] < i]))
                
        pct_cutoff_dict[i] = round(suspicious_price_pct[-1], 3)
       
    return pct_cutoff_dict

In [30]:
find_suspicious_price(data, cutoff=[100, 200, 300, 400, 500, 600, 700, 800, 900, 1000])

{100: np.float64(0.036),
 200: np.float64(0.047),
 300: np.float64(0.055),
 400: np.float64(0.062),
 500: np.float64(0.069),
 600: np.float64(0.076),
 700: np.float64(0.079),
 800: np.float64(0.081),
 900: np.float64(0.083),
 1000: np.float64(0.086)}

## Model baselines

In [31]:
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [32]:
data = data.drop(columns=['region', 'model'])

In [33]:
data = data.dropna(subset=['price'])

In [34]:
X = data.drop(columns=['price'])
y = data['price']

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [36]:
detector = OutlierDetector().fit(X_train, y_train)  # cutoffs learned on train only

# Train: drop price/odometer outliers + out-of-scope rows, keeping X and y aligned
X_train, y_train = detector.transform(X_train, y_train)

# Test: scope rules + price cutoffs learned on train (removes corrupted/fake target labels); odometer not filtered
X_test, y_test = detector.transform(X_test, y_test, filter_price=True, filter_odometer=False)

In [37]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

assert len(numeric_features) + len(categorical_features) == X_train.shape[1], "Feature count mismatch"

In [38]:
def get_model(model_name):
    if model_name == 'linear_regression':
        return LinearRegression()
    elif model_name == 'random_forest':
        return RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1, max_depth=5)
    elif model_name == 'gradient_boosting':
        return XGBRegressor(n_estimators=300, random_state=42, n_jobs=-1, learning_rate=0.05)
    else:
        raise ValueError(f"Model {model_name} not recognized")

In [39]:
def build_pipeline(model_name):
    
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])
    
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
    ])
    
    preprocessor = ColumnTransformer([
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', get_model(model_name))
    ])
    return pipe


def train_and_evaluate_model(model_name, X_train, y_train, n_splits=3):
    pipe = build_pipeline(model_name)
    
    # The whole pipeline is refit on each fold, so imputers/scalers/encoders never see the validation fold
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring={'mse': 'neg_mean_squared_error', 'mae': 'neg_mean_absolute_error', 'r2': 'r2'},
        return_estimator=True
    )
    
    importances = None
    fold_importances = []
    for est in scores['estimator']:
        model = est.named_steps['model']
        names = est.named_steps['preprocessor'].get_feature_names_out()
        if hasattr(model, 'coef_'):
            fold_importances.append(pd.Series(np.ravel(model.coef_), index=names))
        elif hasattr(model, 'feature_importances_'):
            fold_importances.append(pd.Series(model.feature_importances_, index=names))
    if fold_importances:
        importances = pd.concat(fold_importances, axis=1).mean(axis=1).sort_values(ascending=False).head(25)
    
    return {
        'model': model_name,
        'mse': -scores['test_mse'].mean(), 'mse_std': scores['test_mse'].std(),
        'mae': -scores['test_mae'].mean(), 'mae_std': scores['test_mae'].std(),
        'r2': scores['test_r2'].mean(), 'r2_std': scores['test_r2'].std(),
        'feature_importances': importances
    }



def final_test_evaluation(model_name, X_train, y_train, X_test, y_test):
    # Run once, after model/feature choices are final: fit on all of train, score on untouched test
    pipe = build_pipeline(model_name)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    return {
        'model': model_name,
        'mse': mean_squared_error(y_test, y_pred),
        'rmse': mean_squared_error(y_test, y_pred) ** 0.5,
        'mae': mean_absolute_error(y_test, y_pred),
        'r2': r2_score(y_test, y_pred),
        'pipeline': pipe,
    }




In [40]:
#results = train_and_evaluate_model('linear_regression', X_train, y_train)

#results

In [41]:
#results = train_and_evaluate_model('gradient_boosting', X_train, y_train)
#results

In [42]:
#results = train_and_evaluate_model('random_forest', X_train, y_train)
#results

In [43]:
from pathlib import Path
import joblib

RESULTS_DIR = Path(".cache/baseline")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def cached_eval(model_name, X_train, y_train, force=False):
    path = RESULTS_DIR / f"{model_name}.joblib"
    if path.exists() and not force:
        return joblib.load(path)
    result = train_and_evaluate_model(model_name, X_train, y_train)
    joblib.dump(result, path)
    return result

results = {
    name: cached_eval(name, X_train, y_train)
    for name in ['linear_regression', 'random_forest', 'gradient_boosting']
}

### High importance: manufacturer, cylinders, year, odometer, fuel

### Medium importance: type, drive, condition, transmission

### Low importance: Size, paint_color, title_status, state

### Questions: lat, long, model

In [44]:
final_test_evaluation('linear_regression', X_train,  y_train, X_test, y_test)

{'model': 'linear_regression',
 'mse': 102948312.39039576,
 'rmse': 10146.344779791181,
 'mae': 5541.818383296535,
 'r2': 0.46014044064097015,
 'pipeline': Pipeline(steps=[('preprocessor',
                  ColumnTransformer(transformers=[('num',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer()),
                                                                   ('scaler',
                                                                    StandardScaler())]),
                                                   Index(['year', 'odometer', 'lat', 'long'], dtype='str')),
                                                  ('cat',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(strategy='most_frequent')),
                                                                 

In [45]:
final_test_evaluation('gradient_boosting', X_train,  y_train, X_test, y_test)

{'model': 'gradient_boosting',
 'mse': 38381632.0,
 'rmse': 6195.291115032449,
 'mae': 3959.470703125,
 'r2': 0.7987272143363953,
 'pipeline': Pipeline(steps=[('preprocessor',
                  ColumnTransformer(transformers=[('num',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer()),
                                                                   ('scaler',
                                                                    StandardScaler())]),
                                                   Index(['year', 'odometer', 'lat', 'long'], dtype='str')),
                                                  ('cat',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(strategy='most_frequent')),
                                                                   ('encoder',

In [46]:
final_test_evaluation('random_forest', X_train,  y_train, X_test, y_test)

{'model': 'random_forest',
 'mse': 73989769.27025191,
 'rmse': 8601.730597400265,
 'mae': 6017.928442266931,
 'r2': 0.6119986495374454,
 'pipeline': Pipeline(steps=[('preprocessor',
                  ColumnTransformer(transformers=[('num',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer()),
                                                                   ('scaler',
                                                                    StandardScaler())]),
                                                   Index(['year', 'odometer', 'lat', 'long'], dtype='str')),
                                                  ('cat',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(strategy='most_frequent')),
                                                                   ('enc